In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F

sys.path.append("src")
from entropy_pruning import (
    AttentionForecaster,
    UNILoRAClassifier,
    build_attention_cache,
    build_loaders,
    set_seed,
    train_forecaster,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

/home/vcivale/miniconda3/envs/entropy_pruning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [ ]:
CFG = dict(
    data_dir="/raid/DATASETS/NCT-CRC-HE",
    img_size=224,
    batch_size=128,
    num_workers=0,
    seed=42,
    layer_source=2,
    layer_target=23,
    epochs=30,
    lr=1e-4,
    weight_decay=0.05,
    # fixed architecture (best from architecture ablation)
    hidden=256,
    n_heads=4,
    n_layers=2,
    dropout=0.1,
)


# --- Loss functions ---
def loss_kl_mse(pred, target):
    """KL + 0.1*MSE (paper default)."""
    kl = F.kl_div((pred + 1e-8).log(), target + 1e-8, reduction="batchmean")
    mse = F.mse_loss(pred, target)
    return kl + 0.1 * mse


def loss_kl_only(pred, target):
    """KL divergence only."""
    return F.kl_div((pred + 1e-8).log(), target + 1e-8, reduction="batchmean")


def loss_mse_only(pred, target):
    """MSE only."""
    return F.mse_loss(pred, target)


def loss_cosine(pred, target):
    """1 - cosine similarity."""
    return 1 - F.cosine_similarity(pred, target, dim=-1).mean()


def loss_jsd(pred, target):
    """Jensen-Shannon divergence."""
    m = 0.5 * (pred + target)
    kl_pm = F.kl_div((pred + 1e-8).log(), m + 1e-8, reduction="batchmean")
    kl_tm = F.kl_div((target + 1e-8).log(), m + 1e-8, reduction="batchmean")
    return 0.5 * (kl_pm + kl_tm)


LOSS_CONFIGS = [
    ("KL+MSE (paper)", loss_kl_mse),
    ("KL only", loss_kl_only),
    ("MSE only", loss_mse_only),
    ("Cosine", loss_cosine),
    ("JSD", loss_jsd),
]

set_seed(CFG["seed"])
dataset_name = Path(CFG["data_dir"]).name
classifier_ckpt = Path(f"/raid/DATASETS/checkpoints-Attention-Pruning/{dataset_name}/uni_finetuned/best_model.pt")
cache_path = Path(f"/raid/DATASETS/NCT-CRC-HE/data_cache/{dataset_name}_forecaster_dataset.h5")
forecaster_dir = Path(f"/raid/DATASETS/checkpoints-Attention-Pruning//{dataset_name}/loss_ablation")
forecaster_dir.mkdir(parents=True, exist_ok=True)

print(f"Loss functions: {len(LOSS_CONFIGS)}")
for name, _ in LOSS_CONFIGS:
    print(f"  {name}")

Loss functions: 5
  KL+MSE (paper)
  KL only
  MSE only
  Cosine
  JSD


In [4]:
loaders = build_loaders(
    data_dir=CFG["data_dir"],
    img_size=CFG["img_size"],
    batch_size=CFG["batch_size"],
    num_workers=CFG["num_workers"],
    drop_last_train=False,
)

model = UNILoRAClassifier(loaders.n_classes).to(device)
model.load_state_dict(torch.load(classifier_ckpt, map_location=device), strict=False)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

if not cache_path.exists():
    build_attention_cache(
        model=model,
        loaders={"train": loaders.train_loader, "val": loaders.val_loader, "test": loaders.test_loader},
        device=device,
        source_layers=[CFG["layer_source"]],
        target_layers=[CFG["layer_target"]],
        save_path=cache_path,
    )
print("Cache:", cache_path)

del model
torch.cuda.empty_cache()

Cache: /raid/DATASETS/NCT-CRC-HE/data_cache/NCT-CRC-HE_forecaster_dataset.h5


In [ ]:
all_results = []
for i, (loss_name, loss_fn) in enumerate(LOSS_CONFIGS):
    tag = loss_name.replace(" ", "_").replace("(", "").replace(")", "")
    print(f"\n[{i+1}/{len(LOSS_CONFIGS)}] {loss_name}")

    save_path = forecaster_dir / f"forecaster_{tag}.pt"

    forecaster_model = AttentionForecaster(
        embed_dim=1024,
        hidden=CFG["hidden"],
        n_heads=CFG["n_heads"],
        n_layers=CFG["n_layers"],
        dropout=CFG["dropout"],
    )

    result = train_forecaster(
        h5_cache_path=cache_path,
        layer_source=CFG["layer_source"],
        layer_target=CFG["layer_target"],
        device=device,
        epochs=CFG["epochs"],
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"],
        save_path=save_path,
        model=forecaster_model,
        loss_fn=loss_fn,
    )

    result["loss_name"] = loss_name
    all_results.append(result)

    print(f"  rho={result['test_rho_forecaster']:.4f}  "
          f"val_kl={result['best_val_kl']:.4f}")

    del result["model"]
    torch.cuda.empty_cache()


[1/5] KL+MSE (paper)


 79%|███████▊  | 1107/1407 [1:11:14<3:17:51, 39.57s/it]

In [ ]:
df = pd.DataFrame([
    {
        "loss": r["loss_name"],
        "best_val_kl": r["best_val_kl"],
        "best_val_rho": r["best_val_rho"],
        "test_rho_forecaster": r["test_rho_forecaster"],
        "test_rho_token_norm": r["test_rho_token_norm"],
        "delta_rho": r["test_rho_forecaster"] - r["test_rho_token_norm"],
    }
    for r in all_results
])
df = df.sort_values("test_rho_forecaster", ascending=False).reset_index(drop=True)
df.to_csv(f"results/loss_ablation_{dataset_name}.csv", index=False)
df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 1. Bar chart: rho per loss
bars = axes[0].bar(df["loss"], df["test_rho_forecaster"], color="steelblue")
axes[0].set_ylabel("Spearman \u03c1 (test)")
axes[0].set_title("Loss function comparison")
axes[0].bar_label(bars, fmt="%.4f", padding=3)
axes[0].tick_params(axis="x", rotation=25)
axes[0].grid(alpha=0.3, axis="y")

# 2. Bar chart: delta_rho (improvement over norm baseline)
bars2 = axes[1].bar(df["loss"], df["delta_rho"], color="darkorange")
axes[1].set_ylabel("\u0394\u03c1 (forecaster \u2212 norm baseline)")
axes[1].set_title("Improvement over token-norm baseline")
axes[1].bar_label(bars2, fmt="%.4f", padding=3)
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(f"results/loss_ablation_{dataset_name}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("Risultati ablation loss:\n")
print(df[["loss", "test_rho_forecaster", "test_rho_token_norm",
          "delta_rho", "best_val_kl"]].to_string(index=False))